### 7. Qual é a ocorrência de incidentes processuais (como habilitações e impugnações de crédito) por processo de RJ, e como esse volume impacta o gargalo da vara?

**Pergunta:** cada credor que disputa seu crédito gera um "incidente"
processual? E esse volume congestionaria a vara? A pergunta tem duas
camadas: quanto de incidente existe por RJ — e se processos com mais
incidentes tramitam mais devagar.

**Fonte no mart:** `gold_base_analitica` — duas vias de
contagem: (i) os contadores consolidados por processo principal
(`qtd_habilitacoes`, `qtd_impugnacoes`); e (ii) a contagem de processos
autônomos de incidente (classes TPU 111 — Habilitação de Crédito, 114 —
Impugnação de Crédito, e a legada 38) **vinculados à RJ pelo número CNJ**
(os 7 primeiros dígitos do número + o código de origem identificam a
mesma origem processual). O tempo de tramitação vem de
`dias_distribuicao_processamento` (pergunta 1) e
`dias_duracao_processo_total` (ajuizamento → desfecho). Este notebook não
refaz cálculos: consome o resultado pronto.

**Recorte:** classe 129, ajuizado a partir de 09/06/2005.

**Por que faixas e não correlação formal:** estratificar os tempos por
faixas de volume de incidentes (0, 1–5, 6–20, >20) mostra o gradiente do
efeito sem estatística pesada — mesma lógica da pergunta 4.

**Limitações (em linguagem simples):**
1. A contagem depende do vínculo autos-filho → autos-mãe: incidentes em
   autos não vinculados ficam de fora (subcontagem) — e o pool não vinculado
   é majoritariamente do regime anterior à Lei 11.101/2005;
2. Correlação não é causa: RJs maiores (mais credores, mais créditos
   disputados) têm mais incidentes **e** tramitação mais longa por
   complexidade própria — o gradiente mostra associação;
3. Na RJ, a verificação de créditos é internalizada: as impugnações à lista
   do administrador judicial (art. 8º) correm nos próprios autos da RJ e não
   geram processo contável — a métrica capta só a ponta autônoma do conflito
   (habilitação em autos próprios é estrutura típica da falência, art. 119);
4. O stay period não é usado aqui como métrica de gargalo: só é mensurável
   nas convoluções (limitação da pergunta 2).

**O que a consulta a seguir mede:** a ocorrência de incidentes processuais por RJ, por **dupla via independente**:

(i) os contadores consolidados no
mart (`qtd_habilitacoes`, `qtd_impugnacoes`: incidentes computados em autos próprios, classes 111 - habilitação de crédito e 114 - impugnação de crédito, vinculados à RJ-mãe pela chave raiz+origem do número CNJ na construção da gold); e 

(ii) a contagem direta dos processos-filho autônomos (classes 111, 114 e a legada 38 - habilitação)
casados pela mesma chave na hora da consulta. A convergência dos dois totais **valida a regra de vinculação**: se os métodos independentes chegam ao mesmo volume, o vínculo não é artefato.

**Mecânica:** a subconsulta interna agrega os autos-filho por (raiz, origem) e o `LEFT JOIN` devolve o `n_inc` de cada RJ-mãe — quem não tem filho entra como **zero** (`COALESCE`), preservando a semântica contável do universo inteiro. 

A saída traz volume por canal, média, mediana, p90, penetração (% de RJs com ao menos um incidente) e o
extremo. 

Os contadores registram a **ocorrência** do incidente, não
seu conteúdo: não distinguem credor, valor nem desfecho — a leitura é de intensidade da disputa creditícia, não de resultado. A vinculação funciona porque habilitação e impugnação são autuadas **na mesma vara de origem da RJ-mãe** (registro de 1º grau).

**Como os contadores foram apurados (na construção da gold):** `qtd_habilitacoes` e `qtd_impugnacoes` não derivam de
movimentos — cada número conta incidentes autuados em autos próprios (petitórios filhos), vinculados à RJ pelo componente posicional da numeração CNJ (raiz = dígitos 1-7; origem = dígitos 17-20):

- `qtd_habilitacoes` — quantidade de processos filhos de classe 111
  (habilitação de crédito) vinculados à RJ;

- `qtd_impugnacoes` — quantidade de processos filhos de classe 114
  (impugnação à lista de credores) vinculados à RJ.

A cadeia de construção: `fato_incidentes` (1 linha por incidente filho, com gate de unicidade `count = distinct` que elimina duplicidade do mesmo ajuizamento) → agregação `GROUP BY raiz, origem`
→ `LEFT JOIN` na mart pela chave raiz+origem com `COALESCE(..., 0)`:
RJ-mãe sem incidente vinculado entra como zero (não como NULL), o
que preserva a semântica contável.

In [0]:
%sql
-- Q7 · INCIDENCIA (dupla via): contadores do mart + autos-filho autonomos
WITH base AS (
  SELECT r.numero_processo,
         COALESCE(r.qtd_habilitacoes, 0) AS hab,
         COALESCE(r.qtd_impugnacoes, 0)  AS imp,
         COALESCE(f.n_inc, 0)            AS autonomos
  FROM default.gold_base_analitica r
  LEFT JOIN (
    SELECT substring(numero_processo, 1, 7)  AS raiz,
           substring(numero_processo, 17, 4) AS origem,
           count(*)                          AS n_inc
    FROM default.gold_base_analitica
    WHERE classe_codigo IN (111, 114, 38)
    GROUP BY 1, 2
  ) f
    ON f.raiz   = substring(r.numero_processo, 1, 7)
   AND f.origem = substring(r.numero_processo, 17, 4)
  WHERE r.classe_codigo = 129 AND r.escopo_lei_atual
)
SELECT
  count(*)                                                              AS total_rj,
  sum(hab)                                                              AS hab_via_contador,
  sum(imp)                                                              AS imp_via_contador,
  sum(autonomos)                                                        AS incidentes_via_filho,
  round(sum(autonomos) * 1.0 / count(*), 2)                             AS media_incidentes_por_rj,
  round(percentile_approx(autonomos, 0.5), 1)                           AS mediana_incidentes,
  round(percentile_approx(autonomos, 0.9), 1)                           AS p90_incidentes,
  round(count(CASE WHEN autonomos > 0 THEN 1 END) * 100.0
        / count(*), 1)                                                  AS pct_rj_com_incidente,
  max(autonomos)                                                        AS max_incidentes
FROM base;

total_rj,hab_via_contador,imp_via_contador,incidentes_via_filho,media_incidentes_por_rj,mediana_incidentes,p90_incidentes,pct_rj_com_incidente,max_incidentes
5709,180,32,215,0.04,0,0,3.6,2


#### Análise do resultado da incidência

**O que os números dizem:**

| Métrica | Valor | Leitura |
|---|---|---|
| `total_rj` | 5.709 | universo íntegro — igual ao baseline congelado das demais perguntas |
| `hab_via_contador` | 180 | habilitações (classe 111) autuadas em autos próprios, pelo canal consolidado na gold |
| `imp_via_contador` | 32 | impugnações (classe 114) autuadas em autos próprios, pelo mesmo canal |
| `incidentes_via_filho` | 215 | o mesmo fenômeno contado pela via independente (contagem direta dos filhos, inclui a classe legada 38) |
| `media_incidentes_por_rj` | 0,04 | menos de um incidente a cada 25 RJs |
| `mediana_incidentes` | 0 | a RJ típica **não tem nenhum** incidente autônomo |
| `p90_incidentes` | 0 | em 9 de cada 10 RJs, o total de incidentes em autos próprios é **zero** |
| `pct_rj_com_incidente` | 3,6% | apenas ~206 das 5.709 RJs têm ao menos um incidente vinculado |
| `max_incidentes` | 2 | nenhuma RJ chega a 3 incidentes — não existe cauda |

**A validação da dupla via.** Os dois métodos independentes convergem: o canal consolidado na gold soma 180 + 32 = **212**, e a contagem direta dos autos-filho soma **215** — diferença de 3, explicada pelo critério de cada via: os contadores do mart recortam apenas as classes 111 e 114, enquanto a contagem direta inclui a classe 38 (legada,
regime anterior à lei). A convergência em dois canais construídos de forma independente **valida a regra de vinculação raiz+origem** para incidentes: o vínculo não é artefato da modelagem, e o resíduo de 3 é atribuível, não inexplicável.

**O achado e sua limitação.**

**Achado:** habilitações e impugnações autuadas como processos
próprios são raras nas RJs do corpus — 3,6% de penetração (212
incidentes em classes 111/114; 215 com a classe legada 38), máximo de
2 por RJ, mediana 0.

**Limitação do achado:** essa métrica conta apenas o incidente
**autuado em autos próprios** (processo-filho com número CNJ próprio).
Mas na prática forense o incidente raramente ganha autos próprios: o
credor divergente apresenta sua discordância como **petição dentro
dos próprios autos da RJ** (após a publicação da lista de credores —
art. 8º da Lei 11.101/2005), e o juiz decide nos mesmos autos. O dado
não registra isso como processo separado — portanto o contador não
vê.

**Consequência para a resposta:** o 3,6% não mede a frequência da
disputa creditícia — mede apenas a **ponta visível** dela, a que
migrou para autos próprios. A frequência real é maior, mas
invisível neste formato de dado (mesma limitação já documentada na
homologação do plano — pergunta 2 — e no vínculo recursal — pergunta 9).

**Coerência interna da distribuição:** a comparação média × mediana
(0,04 × 0) confirma distribuição degenerada — quase todo o universo em
zero, sem cauda expressiva (o máximo de 2 descarta concentração em
poucas RJs). O número é pequeno e homogêneo, não concentrado — o que
também limita a utilidade estatística de qualquer corte por grupo
(poucos pares para comparar).

**Implicação para a resposta da pergunta 7 (parte 1):** o canal
mensurável no dado estruturado é o dos autos-filho, e ele registra
**3,6% de penetração** (212 incidentes pelas classes 111/114; 215
incluindo a legada 38; máximo de 2 por RJ). A resposta final deve
apresentar o número **com a ressalva de canal**: mede-se a incidência
de incidentes autuados em autos próprios, não a frequência real das
habilitações e impugnações, que no registro migram majoritariamente
para movimentos dentro dos autos da RJ.

**O que a consulta a seguir mede:** o **impacto no gargalo da vara** — compara
os tempos de tramitação das RJs **com** ao menos um incidente autônomo
vinculado contra as **sem** incidente: mediana de dias até o
processamento (pergunta 1) e mediana da duração total do processo
(ajuizamento → desfecho). Se incidentes congestionam a vara, o grupo
com incidente deve tramitar mais devagar.

**Mecânica:** os incidentes são reduzidos ao par (raiz, origem)
deduplicado; o `LEFT JOIN` marca cada RJ-mãe como 'com incidente' ou
'sem incidente'; cada grupo é agregado com medianas (robustas a
extremos, padrão do projeto). `com_proc` e `com_desfecho` reportam os
`n` de cada lado — a mediana de processamento só existe para quem tem
o marco, e omitir os `n` esconderia viés de seleção.

**Aviso metodológico:** o grupo com incidente é pequeno (~208 RJs,
poucos desfechos registrados) — diferenças de medianas entre grupos
desse tamanho são **indício, não causalidade**.

In [0]:
%sql
-- Q7 · CONTRASTE: tempos das RJs com vs sem incidente autonomo
WITH inc AS (
  SELECT substring(numero_processo, 1, 7)  AS raiz,
         substring(numero_processo, 17, 4) AS origem
  FROM default.gold_base_analitica
  WHERE classe_codigo IN (111, 114, 38)
),
base AS (
  SELECT CASE WHEN i.raiz IS NOT NULL THEN 'com incidente' ELSE 'sem incidente' END AS grupo,
         r.dias_distribuicao_processamento,
         r.dias_duracao_processo_total
  FROM default.gold_base_analitica r
  LEFT JOIN (SELECT DISTINCT raiz, origem FROM inc) i
         ON i.raiz = substring(r.numero_processo, 1, 7)
        AND i.origem = substring(r.numero_processo, 17, 4)
  WHERE r.classe_codigo = 129 AND r.escopo_lei_atual
)
SELECT grupo,
       count(*)                                                          AS qtd_rj,
       count(dias_distribuicao_processamento)                            AS com_proc,
       round(percentile_approx(dias_distribuicao_processamento, 0.5), 1) AS mediana_proc,
       count(dias_duracao_processo_total)                                AS com_desfecho,
       round(percentile_approx(dias_duracao_processo_total, 0.5), 1)     AS mediana_duracao_total
FROM base
GROUP BY 1;

grupo,qtd_rj,com_proc,mediana_proc,com_desfecho,mediana_duracao_total
com incidente,208,63,52,24,1459
sem incidente,5501,2359,48,1013,1467


#### Análise do resultado do contraste (com × sem incidente)

**O que os números dizem:**

| Métrica | Com incidente | Sem incidente | Leitura |
|---|---|---|---|
| `qtd_rj` | 208 | 5.501 | o grupo com incidente é 3,6% do universo — confere com a célula anterior |
| `com_proc` | 63 (30%) | 2.359 (43%) | RJs com incidente têm menos vezes o marco do processamento datado |
| `mediana_proc` | 52 dias | 48 dias | diferença de 4 dias (~8%) — pequena |
| `com_desfecho` | 24 | 1.013 | a mediana de duração do grupo com incidente repousa sobre **24 casos** |
| `mediana_duracao_total` | 1.459 dias | 1.467 dias | diferença de 8 dias — nula na prática (e favorável ao grupo com incidente) |

**O achado:** não há diferença relevante de tramitação entre as RJs
com e sem incidente autuado em autos próprios. A mediana até o
processamento diverge em apenas 4 dias (52 × 48) e a duração total é
praticamente idêntica (1.459 × 1.467) — na direção **oposta** à
esperada pelo gargalo, o grupo com incidente tramita (leve e
irrelevantemente) mais rápido.

**A limitação do achado:** o grupo com incidente é pequeno e a
comparação repousa sobre amostras mínimas — a mediana de duração usa
**24 casos** e a de processamento, **63**. Diferenças de poucos dias
em amostras desse tamanho não sustentam qualquer inferência; e as
proporções de `com_proc` (30% × 43%) e `com_desfecho` (11% × 18%)
sugerem composição temporal distinta entre os grupos (as RJs com
incidente vinculado são, em média, de ajuizamento diferente), o que
contamina qualquer comparação direta de medianas.

**Consequência para a resposta (parte 2 da pergunta 7):** o canal
mensurável — incidentes autuados em autos próprios, 3,6% de
penetração — é **raro demais e homogêneo demais para produzir efeito
detectável no gargalo da vara**. A resposta final deve registrar: (i)
o volume de incidentes por autos-filho é ínfimo e não se associa a
tramitação mais lenta neste dado; (ii) a leitura não exclui impacto
real, porque a maior parte da disputa creditícia corre dentro dos
autos da RJ (petições e decisões internas), formato invisível ao
contador — mesma fronteira estrutural documentada na parte 1 e nas
perguntas 2 e 9.

#### Resposta final da pergunta 7
**Parte 1 — Ocorrência de incidentes por RJ.**

No canal mensurável do dado estruturado — incidentes autuados em
**autos próprios** (processos-filho de habilitação, classe 111, e
impugnação, classe 114, vinculados pela chave raiz+origem, validado
por dupla via independente: 212 pelo contador consolidado × 215 pela
contagem direta, diferença atribuível à classe legada 38) — a
ocorrência é mínima:

- **212 incidentes autônomos em 5.709 RJs** (180 habilitações + 32
  impugnações);
- **3,6% de penetração**: apenas ~208 RJs têm ao menos um incidente;
- mediana **0** e p90 **0**: a RJ típica não tem nenhum incidente
  autônomo — e nenhuma RJ passa de 2.

**Parte 2 — Impacto no gargalo da vara.**

Comparando RJs com × sem incidente autônomo (208 × 5.501), **não há
diferença relevante de tramitação**: mediana até o processamento de 52
dias (com incidente) × 48 dias (sem), e duração total praticamente
idêntica (1.459 × 1.467 dias) — na direção oposta à esperada pelo
gargalo. O comparativo repousa sobre amostras mínimas (63 marcos e 24
desfechos no grupo com incidente), o que reforça a leitura de
**ausência de efeito detectável**, não a prova de ausência de efeito.

**Limitação estrutural que condiciona ambas as partes.** O contador só
enxerga o incidente autuado como processo autônomo. Na prática
forense, o incidente raramente ganha autos próprios: o credor
divergente apresenta sua discordância como petição dentro dos
próprios autos da RJ (após a publicação da lista de credores —
art. 8º da Lei 11.101/2005), e o juiz decide nos mesmos autos — sem
processo novo, sem número CNJ próprio, invisível ao contador. O
número medido é a ponta visível da disputa creditícia, não sua
frequência real.

**Síntese.** O dado estruturado mostra incidentes em autos próprios raros (3,6% das RJs; máximo de 2 por RJ) sem qualquer associação a tramitação mais lenta. O número baixo, porém, não significa que a disputa de créditos seja rara: significa que a maior parte dela não vira processo separado — corre dentro dos próprios autos da RJ, como petição e decisão nos mesmos autos, formato que o
padrão Datajud não registra com número próprio.
